In [ ]:
%pip install openai anthropic --upgrade --quiet

# Understanding Prompt Caching in OpenAI and Anthropic

This notebook demonstrates how to:
1. Monitor OpenAI's automatic prompt caching
2. Implement manual prompt caching with Anthropic's API
3. Compare the cost savings between different approaches

In [1]:
import os
from getpass import getpass
from openai import OpenAI
import anthropic
import time

In [2]:
# Model Configuration
OPENAI_MODEL = "gpt-5-mini"
ANTHROPIC_MODEL = "claude-3-haiku-20240307"  # Change to a newer model if available on your API key

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

openai_client = OpenAI()
anthropic_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

## Part 1: OpenAI Automatic Prompt Caching (Responses API)

OpenAI automatically caches prompts longer than 1,024 tokens. The Responses API uses `instructions=` for system prompts and `input=` for user messages. Cache usage is reported in `response.usage.prompt_tokens_details.cached_tokens`.

In [3]:
def check_openai_caching(system_prompt, user_prompt):
    """Make an OpenAI Responses API call and check cache usage.
    
    OpenAI automatically caches prompts >1024 tokens. The cached_tokens
    field in usage.input_tokens_details shows how many tokens were served
    from cache.
    """
    response = openai_client.responses.create(
        model=OPENAI_MODEL,
        instructions=system_prompt,
        input=user_prompt
    )
    
    usage = response.usage
    cached_tokens = 0
    if usage.input_tokens_details:
        cached_tokens = usage.input_tokens_details.cached_tokens or 0
    
    print(f"Input tokens:  {usage.input_tokens}")
    print(f"Cached tokens: {cached_tokens} ({cached_tokens / usage.input_tokens * 100:.0f}% cache hit)" if cached_tokens else f"Cached tokens: 0 (cache miss)")
    print(f"Output tokens: {usage.output_tokens}")
    print(f"Total tokens:  {usage.total_tokens}")
    
    return response

Let's test it with a long system prompt:

In [5]:
# Create a long system prompt (>2048 tokens to meet both OpenAI and Anthropic Haiku minimums)
long_system_prompt = """You are an AI assistant specialized in explaining complex topics. Here are 20 examples of good explanations that demonstrate the ideal structure, depth, and clarity we aim for:

1. EXAMPLE - Quantum Computing Example:
When explaining quantum computing, start with classical bits (0s and 1s) and contrast them with quantum bits (qubits) that can exist in multiple states simultaneously through superposition. Use the analogy of a coin: a classical bit is like a coin lying flat (definitely heads or tails), while a qubit is like a spinning coin (simultaneously heads and tails until observed). Emphasize how quantum entanglement allows qubits to be interconnected in ways classical bits cannot, exponentially increasing computational power for specific tasks.

2. EXAMPLE - Photosynthesis:
Break down photosynthesis into its key stages: light-dependent reactions and the Calvin cycle. Describe how chlorophyll molecules capture sunlight energy, converting it into chemical energy (ATP and NADPH). Then explain how this energy is used to convert CO2 into glucose through the Calvin cycle. Use the analogy of a solar-powered factory where sunlight is the power source (light-dependent reactions) and the assembly line (Calvin cycle) builds sugar molecules from carbon dioxide.

3. EXAMPLE - Special Relativity:
Begin with Einstein's two postulates: the constancy of light speed and the equivalence of physical laws in all inertial reference frames. Use the classic train-and-platform thought experiment to illustrate simultaneity. Explain time dilation using the light clock analogy, showing why moving clocks tick more slowly from a stationary observer's perspective. Connect these concepts to the famous equation E=mc², explaining how mass and energy are different forms of the same thing.

4. EXAMPLE - Neural Networks:
Compare artificial neural networks to biological brains: neurons are like nodes, synapses like weighted connections. Explain how information flows from input layer through hidden layers to output, with each connection being strengthened or weakened during training (like building muscle memory). Use the analogy of a complex voting system where each neuron "votes" on the final output, with their votes weighted by connection strengths learned from training data.

5. EXAMPLE - Climate Change:
Start with the greenhouse effect basics: certain gases (CO2, methane) trap heat like glass in a greenhouse. Explain how human activities increase these gases' concentration, enhancing the natural greenhouse effect. Use the analogy of adding blankets to a bed - each additional blanket (greenhouse gas) makes it harder for heat to escape. Connect this to observable changes: rising temperatures, sea levels, and extreme weather events.

6. EXAMPLE - DNA Replication:
Compare DNA replication to unzipping a zipper and creating two new zippers, each with one old side and one new side. Detail how helicase "unzips" the double helix, then polymerase adds complementary nucleotides to each strand. Emphasize the importance of base pairing (A-T, C-G) and how this ensures accurate copying. Use the analogy of a document being photocopied, where each half serves as a template for rebuilding the whole.

7. EXAMPLE - Black Holes:
Describe black holes as regions where gravity is so strong that nothing, not even light, can escape beyond the event horizon. Use the analogy of a river flowing toward a waterfall - past a certain point (event horizon), the current is too strong for anything to swim back upstream. Explain how black holes form from collapsed massive stars and how they affect space-time, like a heavy ball creating a deep depression in a stretched rubber sheet.

8. EXAMPLE - Blockchain:
Compare blockchain to a public ledger that everyone can see and verify but nobody can alter without consensus. Each block contains transactions and links to the previous block (like a chain of sealed evidence bags in a criminal case). Explain how mining works through the analogy of a complex puzzle that requires significant effort to solve but is easy to verify once solved. Emphasize how this system creates trust without requiring a central authority.

9. EXAMPLE - Immune System:
Describe the immune system as a sophisticated defense network with multiple layers: physical barriers (skin, mucus), general defenders (white blood cells), and specialized units (antibodies). Use the analogy of a castle's defense system with walls (barriers), guards (innate immunity), and specially trained soldiers (adaptive immunity). Explain how vaccines work by training these defenses to recognize specific threats before they become dangerous.

10. EXAMPLE - Evolution:
Explain natural selection through the analogy of a sieve: environmental pressures "filter out" less advantageous traits while allowing beneficial ones to pass through to future generations. Describe how random mutations provide the raw material for selection, like a lottery where most tickets (mutations) lose but occasional winners spread through the population. Emphasize the timescale involved and how small changes accumulate over many generations.

11. EXAMPLE - Quantum Entanglement:
Describe entanglement as a special correlation between particles: measuring one instantly determines the state of the other, regardless of distance. Use the analogy of a pair of magic dice that always show matching numbers when rolled simultaneously, even on opposite sides of the planet. Clarify that this does not allow faster-than-light communication, as you cannot control which outcome appears - only observe the correlation after the fact.

12. EXAMPLE - Plate Tectonics:
Explain that Earth's surface is broken into massive plates floating on semi-molten rock. Use the analogy of cracked eggshell pieces sliding on the white beneath. Describe how convection currents in the mantle drive plate movement, creating earthquakes at boundaries, mountains where plates collide, and new ocean floor where they pull apart. Connect this to the observation that continents fit together like puzzle pieces.

13. EXAMPLE - The Human Digestive System:
Compare the digestive system to a disassembly line: food enters as complex structures and is progressively broken into smaller molecules the body can absorb. Walk through each stage - mouth (mechanical breakdown), stomach (acid bath), small intestine (nutrient absorption with massive surface area from villi), and large intestine (water recovery). Use the analogy of a recycling plant that sorts, crushes, and extracts valuable materials from mixed waste.

14. EXAMPLE - The Big Bang Theory:
Start by clarifying that the Big Bang was not an explosion in space but an expansion of space itself. Use the analogy of dots on a balloon: as the balloon inflates, every dot moves away from every other dot, with no single center. Explain the evidence - cosmic microwave background radiation (the afterglow), redshift of distant galaxies, and the abundance of light elements - and how they all point to an initially hot, dense state roughly 13.8 billion years ago.

15. EXAMPLE - Machine Learning Algorithms:
Distinguish supervised, unsupervised, and reinforcement learning using a classroom analogy. Supervised learning is like a student with an answer key studying for a test. Unsupervised learning is like sorting a box of unlabeled objects into groups by similarity. Reinforcement learning is like training a dog with treats and corrections - the agent learns by trial and error which actions yield rewards.

16. EXAMPLE - Cellular Respiration:
Present cellular respiration as the reverse of photosynthesis: cells break down glucose to release stored energy as ATP. Use the analogy of burning fuel in an engine - glucose is the fuel, oxygen is needed for combustion, and ATP is the usable energy output. Walk through glycolysis, the Krebs cycle, and the electron transport chain as three stages of increasingly efficient energy extraction.

17. EXAMPLE - Wave-Particle Duality:
Explain that light and matter exhibit both wave-like and particle-like behavior depending on how they are observed. Use the double-slit experiment: when not observed, particles create interference patterns like waves; when observed, they behave like discrete particles. Compare this to a person who behaves differently when they know they are being watched - the act of measurement fundamentally changes the outcome.

18. EXAMPLE - The Human Nervous System:
Compare the nervous system to a communication network: the brain is the central server, the spinal cord is the main cable, and nerves are the wires reaching every part of the body. Explain how electrical signals (action potentials) travel along neurons and chemical signals (neurotransmitters) bridge the gaps between them. Use the analogy of a relay race where the baton is passed at each synapse.

19. EXAMPLE - Chemical Bonding:
Explain that atoms bond because they seek stable electron configurations. Ionic bonds are like a transaction - one atom donates electrons to another (like giving away extra coins to someone who needs them). Covalent bonds are like sharing - atoms pool electrons to fill their outer shells (like roommates splitting rent). Metallic bonds are like a communal pool where electrons flow freely among all atoms, explaining why metals conduct electricity.

20. EXAMPLE - Nuclear Fusion:
Describe fusion as forcing light nuclei together to form heavier ones, releasing enormous energy. Use the analogy of snapping strong magnets together - it takes effort to push them close, but once they click, the bond releases energy. Explain why the Sun fuses hydrogen into helium, why fusion is so hard to achieve on Earth (extreme temperature and pressure needed), and why it promises nearly unlimited clean energy if we can sustain it.

For each explanation, remember to:
1. Start with familiar concepts and build toward complex ones
2. Use clear, vivid analogies that connect to everyday experience
3. Break down complex processes into understandable steps
4. Address common misconceptions proactively
5. Provide real-world applications and examples
6. Maintain scientific accuracy while ensuring accessibility
7. Include relevant visualizations or diagrams when helpful
8. Connect the topic to related fields and broader implications

When responding to questions:
- First assess the questioner's current understanding level
- Choose appropriate analogies from their field of knowledge
- Build explanations iteratively, checking for comprehension
- Highlight key concepts and their interconnections
- Anticipate and address likely follow-up questions
- Provide concrete examples and practical applications
- Maintain engagement through conversational tone
- End with a clear summary of main points"""

# First call - may or may not hit cache depending on prior requests
print("First call:")
response1 = check_openai_caching(long_system_prompt, "Explain quantum computing")

time.sleep(2)  # Small delay between calls

# Second call - should show caching (same prefix)
print("\nSecond call:")
response2 = check_openai_caching(long_system_prompt, "Explain neural networks")

First call:
Input tokens:  2066
Cached tokens: 0 (cache miss)
Output tokens: 1529
Total tokens:  3595

Second call:
Input tokens:  2066
Cached tokens: 1792 (87% cache hit)
Output tokens: 1417
Total tokens:  3483


## Part 2: Anthropic Manual Prompt Caching

With Anthropic, we can explicitly control what gets cached using the `cache_control` parameter:

In [6]:
def create_anthropic_cached_message(system_content, user_message):
    """Create a message with cached system content in Anthropic.
    
    Prompt caching is now built into the standard messages.create() API.
    Use cache_control on content blocks to control what gets cached.
    """
    response = anthropic_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1024,
        system=[
            {
                "type": "text",
                "text": system_content,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[{"role": "user", "content": user_message}]
    )
    
    # Print usage statistics
    usage = response.usage
    print(usage)
    print(f"Cache creation tokens: {usage.cache_creation_input_tokens}")
    print(f"Cache read tokens: {usage.cache_read_input_tokens}")
    print(f"Regular input tokens: {usage.input_tokens}")
    print(f"Output tokens: {usage.output_tokens}")
    
    return response

In [7]:
# Test Anthropic caching - using the same system prompt as the OpenAI example
long_context = long_system_prompt

print("First call (should create cache):")
response1 = create_anthropic_cached_message(long_context, "Analyze this text.")

time.sleep(2)

print("\nSecond call (should use cache):")
response2 = create_anthropic_cached_message(long_context, "Summarize the main points.")

First call (should create cache):
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=2277), cache_creation_input_tokens=2277, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=11, output_tokens=714, server_tool_use=None, service_tier='standard')
Cache creation tokens: 2277
Cache read tokens: 0
Regular input tokens: 11
Output tokens: 714

Second call (should use cache):
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=2277, inference_geo='not_available', input_tokens=13, output_tokens=254, server_tool_use=None, service_tier='standard')
Cache creation tokens: 0
Cache read tokens: 2277
Regular input tokens: 13
Output tokens: 254


## Part 3: Cost Analysis

Let's create a function to calculate the cost savings from caching:

In [8]:
def calculate_cost_savings(cached_tokens, model):
    """Calculate cost savings from using cached tokens.
    
    Pricing (per million tokens):
    - Claude 3 Haiku: $0.25/MTok input, $0.03/MTok cache read (10% of base)
    - GPT-5 mini: $0.25/MTok input, $0.025/MTok cached input (10% of base)
    
    Update the prices dict below if using a different Anthropic model.
    """
    prices = {
        "claude-3-haiku-20240307": {
            "name": "Claude 3 Haiku",
            "base_input": 0.25 / 1_000_000,      # $0.25/MTok
            "cache_read": 0.03 / 1_000_000,       # $0.03/MTok (10% of base)
        },
        "claude-sonnet-4-5-20250514": {
            "name": "Claude Sonnet 4.5",
            "base_input": 3.00 / 1_000_000,       # $3/MTok
            "cache_read": 0.30 / 1_000_000,       # $0.30/MTok (10% of base)
        },
        "gpt-5-mini": {
            "name": "GPT-5 mini",
            "base_input": 0.25 / 1_000_000,       # $0.25/MTok
            "cache_read": 0.025 / 1_000_000,      # $0.025/MTok (10% of base)
        }
    }
    
    model_prices = prices.get(model, prices["gpt-5-mini"])
    base_cost = cached_tokens * model_prices["base_input"]
    cached_cost = cached_tokens * model_prices["cache_read"]
    savings = base_cost - cached_cost
    
    print(f"Model: {model_prices['name']}")
    print(f"Base cost for {cached_tokens:,} tokens: ${base_cost:.6f}")
    print(f"Cost with caching: ${cached_cost:.6f}")
    print(f"Savings: ${savings:.6f} ({(savings/base_cost)*100:.1f}%)")
    
    return savings

In [9]:
# Example cost analysis for 10,000 cached tokens
print("Anthropic Claude Sonnet 4.5:")
anthropic_savings = calculate_cost_savings(10000, ANTHROPIC_MODEL)

print("\nOpenAI GPT-5 mini:")
openai_savings = calculate_cost_savings(10000, OPENAI_MODEL)

Anthropic Claude Sonnet 4.5:
Model: Claude 3 Haiku
Base cost for 10,000 tokens: $0.002500
Cost with caching: $0.000300
Savings: $0.002200 (88.0%)

OpenAI GPT-5 mini:
Model: GPT-5 mini
Base cost for 10,000 tokens: $0.002500
Cost with caching: $0.000250
Savings: $0.002250 (90.0%)


-------------------------

## Best Practices and Tips

1. **Minimum Token Requirements**:
   - OpenAI: >1,024 tokens (automatic caching)
   - Anthropic: >1,024 tokens for Sonnet/Opus, >2,048 tokens for Haiku

2. **Cache Lifetime**:
   - OpenAI: 5-10 minutes (automatic, no manual control)
   - Anthropic: 5-minute (default) or 1-hour cache durations (`"type": "ephemeral"`)

3. **How Caching Works**:
   - **OpenAI**: Automatic — prompts >1,024 tokens are cached automatically. Cached input costs 10% of base price. No code changes needed.
   - **Anthropic**: Manual control via `cache_control` on content blocks in the standard `messages.create()` API. Cache reads cost 10% of base price; 5-min cache writes cost 1.25x base; 1-hour cache writes cost 2x base.

4. **Optimization Strategies**:
   - Put static content (instructions, context, examples) at the beginning
   - Keep dynamic content separate from cached content
   - Monitor cache hit rates to optimize your implementation

5. **Common Use Cases**:
   - Long system prompts with examples
   - Document analysis with consistent context
   - Multi-turn conversations with stable history
   - Code analysis with repository context

## Summary

- **OpenAI** offers automatic prompt caching with no code changes — prompts >1,024 tokens are cached for 5-10 minutes, with cache hits charged at 10% of the base input price.
- **Anthropic** provides explicit cache control via `cache_control` blocks in the standard `messages.create()` API, with 5-minute or 1-hour cache durations and cache reads at 10% of base price.
- Both providers offer 90% savings on cached token reads, making caching highly cost-effective for repeated prompts.